### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Adjust to point to the actual root of your project
PROJECT_ROOT = Path.cwd().parent  # or Path("/absolute/path/to/your/project")
sys.path.insert(0, str(PROJECT_ROOT))

### Loading the model

In [2]:
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

model_name = "deepseek-ai/DeepSeek-V2-Lite"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map='auto', 
    attn_implementation='eager',  
    trust_remote_code=True
)
model.generation_config = GenerationConfig.from_pretrained(model_name)
model.generation_config.pad_token_id = model.generation_config.eos_token_id
model.eval()

text = "The goal of life is to"
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs.to(model.device), max_new_tokens=10)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

/home/p84400019/miniconda3/envs/int/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:57<00:00, 14.35s/it]
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


[MoE Inference] new_x.shape=torch.Size([42, 2048]), topk_ids.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_before_mul.shape=torch.Size([7, 6, 2048]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_after_mul.shape=torch.Size([7, 6, 2048])
[MoE Inference] summed_out.shape=torch.Size([7, 2048])
[MoE Inference] y.shape=torch.Size([1, 7, 2048]), topk_idx.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] y.shape=torch.Size([1, 7, 2048])
[MoE Inference] new_x.shape=torch.Size([42, 2048]), topk_ids.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_before_mul.shape=torch.Size([7, 6, 2048]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] out_after_mul.shape=torch.Size([7, 6, 2048])
[MoE Inference] summed_out.shape=torch.Size([7, 2048])
[MoE Inference] y.shape=torch.Size([1, 7, 2048]), topk_idx.shape=torch.Size([7, 6]), topk_weight.shape=torch.Size([7, 6])
[MoE Inference] y.shape=tor

### Record activations, save them, and verify them

In [3]:
from src.activation_recorder import ActivationRecorder, MultiPromptActivations

max_new_tokens=10
recorder = ActivationRecorder(model, tokenizer)
prompts = ["Hello, world! How can you code",  "Tell me a joke"]
activations = recorder.record_prompts(prompts, max_new_tokens=max_new_tokens)
recorder.verify_recorded_activations(activations, diff_q_size=True)

# Save the activations to disk.
save_dir = "../data/activations"
activations.save(save_dir)

# Load the activations from disk.
file_path = os.path.join(save_dir, "multi_prompt_activations.pkl")
loaded_activations = MultiPromptActivations.load(file_path)

# Optional: verify the loaded activations match the saved ones.
print("Loaded MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Check again the activations
recorder.verify_recorded_activations(loaded_activations, diff_q_size=True)

print("Final MultiPromptActivations object has:", len(activations.prompts), "prompts recorded.")

# Extract the first prompt, first step, first layer, first head
prompt_acts = activations.prompts[0]
step_acts = prompt_acts.steps[0]
layer_acts = step_acts.layers[0]
attn = layer_acts.attention
for head_acts in attn.heads:
    print(head_acts.query.shape)
    print(head_acts.attention_weights.shape)
    print(head_acts.attention_outputs.shape)
    print(head_acts.projected_outputs.shape)


ModuleNotFoundError: No module named 'activation_recorder'